## Build a Wflow model from scratch using Python

This notebook demonstrates how to build a **Wflow** model from scratch, step by step, using Python.

Unlike the [command line interface (CLI) example](build_model_cli.ipynb), where the full model is built with a single command, we run each setup step separately. This lets you inspect the result after each step, which makes it easier to understand what each step does and to find problems early.

Let's first load the required libraries

In [ ]:
import numpy as np
import pandas as pd

# For plotting
from matplotlib import colors
import matplotlib.pyplot as plt

# Import HydroMT and HydroMT-Wflow
import hydromt
from hydromt.model.processes.basin_mask import get_basin_geometry
from hydromt_wflow import WflowSbmModel

### The data catalog
For this example, we will use the artifact data for our DataCatalog, which contains data for the Piave basin in northern Italy.

Let's have a look at the data contained in the artifact data:

In [ ]:
data_catalog = hydromt.DataCatalog(data_libs="artifact_data")
data_catalog

That's a lot of data! The DataCatalog lists for each entry what kind of data it is, how it is formatted, what the CRS is, what the source is, etc. 

HydroMT-Wflow expects for each data variable a specific name and unit. For example, elevation data should always be named `elevtn` and have a unit of m+REF. Of course, different datasets use widely different units and naming. With the DataCatalog, we can take care of this, by specifying for each datasets how variables should be mapped, and how units should be converted.

Let's checkout one of the data sources, for example MERIT Hydro:

In [ ]:
ds_merit_hydro = data_catalog.get_rasterdataset('merit_hydro')
ds_merit_hydro

MERIT Hydro is a dataset containing both topographic (elevation, slope) and hydrographic (flow direction, upstream area) data. Let's plot the elevation:

In [ ]:
# Define a nice colourmap
c_dem = plt.cm.terrain(np.linspace(0.25, 1, 256))
cmap = colors.LinearSegmentedColormap.from_list("dem", c_dem)

fig, ax = plt.subplots(figsize=(6, 6))

# Plot the elevation data, masking nodata values
ds_merit_hydro['elevtn'].raster.mask_nodata().plot(
    ax=ax,
    cmap=cmap,
    vmax=2500,
    cbar_kwargs={'label': 'Elevation [m]'})
ax.set_xlabel('Longitude [degrees]')
ax.set_ylabel('Latitude [degrees]')
ax.set_title('MERIT Hydro Elevation')
ax.set_aspect('equal');

### Defining the model region
When building a Wflow model with HydroMT-Wflow, we need to specify for which geographic region we want to build a model. Very often, our Wflow model area is based on a (sub)basin. For example, we can specify an outlet location of our Wflow model, and HydroMT-Wflow will derive the area upstream of this outlet, which then defines the geographic region. It is also possible to define multiple outlets, or even specify an entire area. HydroMT-Wflow will derive all (sub)basins belonging to the specified outlets or area. A Wflow model can thus contain multiple basins! For more details, see the [HydroMT documentation on this](https://deltares.github.io/hydromt/latest/user_guide/models/model_region.html).

In this notebook, we will specify a single outlet, and derive the corresponding upstream area. For HydroMT-Wflow to be able to derive the upstream area, and build the Wflow model, it needs certain hydrographic data, derived from a Digital Elevation Model (DEM).

In this example, we assume this hydrographic data has already been derived. In fact, our data catalog with artifact data already contains datasets with variables such as the local drainage direction and upstream area. If you want to know how to derive this hydrographic data from a raw DEM, please see this [example notebook on how to prepare flow directions and related data from a raw DEM](prepare_ldd.ipynb).

In [ ]:
# Outlet coordinates
x = 12.2051
y = 45.8331

# Derive the basin geometry and outlet location
gdf_basin, gdf_outlet = get_basin_geometry(
    ds=data_catalog.get_rasterdataset('merit_hydro'), # hydrography dataset from the data catalog
    kind="subbasin",
    xy=[x, y],
    uparea=70) # set a minimum upstream area to force snapping to a river

Let's explain briefly what we just did. We supply an outlet location `[x, y]`, and tell HydroMT-Wflow to delineate the subbasin upstream of this outlet. We also tell it to use MERIT Hydro as its hydrography dataset, but we could have used another dataset as well. Lastly, we tell it that the area upstream of our outlet should be at least 70 km2. If the cell at our supplied outlet does not meet this requirement, HydroMT-Wflow will snap it to the closest cell that does. By setting `kind="subbasin"` it will delineate only the area upstream of our outlet. If we had used `kind="basin"`, it would have returned the entire basin, also downstream of our `[x, y]`.

Let's plot the delineated subbasin!

In [ ]:
# Plot the elevation data and the delineated subbasin and outlet
fig, ax = plt.subplots(figsize=(8, 8))

ds_merit_hydro['elevtn'].raster.mask_nodata().plot(
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=2500,
    cbar_kwargs={'label': 'Elevation [m]', 'shrink': 0.7})

# Plot the boundary of the delineated  subbasin
gdf_basin.boundary.plot(ax=ax, edgecolor="k", zorder=2, label="Delineated subbasin")

# Plot the outlet of the delineated subbasin
gdf_outlet.plot(ax=ax, markersize=40, c="red", zorder=2, label="Outlet")

ax.set_xlabel('Longitude [degrees]')
ax.set_ylabel('Latitude [degrees]')
ax.set_title('MERIT Hydro Elevation')
ax.set_aspect('equal')
ax.legend(loc='lower right');

Also note that the outlet has slightly shifted from our original `[x, y]`:

In [ ]:
print(f'Original x-coordinate: {x:.4f} - Outlet x-coordinate: {gdf_outlet["geometry"].x.values[0]:.4f}')
print(f'Original y-coordinate: {y:.4f} - Outlet y-coordinate: {gdf_outlet["geometry"].y.values[0]:.4f}')

This is because we forced snapping to a location with an upstream area of at least 70 km2.

### Initializing our model
We just used HydroMT-Wflow to delineate a subbasin based on an outlet location. However, note that we haven't built any model whatsoever, yet. This is just a handy step to check we correctly defined the region of our interest.

Next, we can use this geographic region of interest to build our Wflow model.

We start with a completely empty Wflow model. We only specify the directory where the model should be stored, and the data catalog configuration files (for now just the artifact data).

In [ ]:
# This creates a completely empty Wflow model.
model = WflowSbmModel("./wflow_test_python_full",
                      data_libs="artifact_data")

As you can see, the model is still empty:

In [ ]:
print("Static model data:")
display(model.staticmaps.data)
print("Model forcing data:")
display(model.forcing.data)
print("Model geometry data:")
display(model.geoms.data)

Let's start by configuring some model settings, such as the start and end time, and the timestep. Many model settings have a default value, or are adjusted automatically when we run certain workflows. For example, glaciers are by default turned off, but once we add glaciers to our model, our model settings will automatically be updated to turn the glaciers on.

You can always easily adjust the settings of your Wflow model in the Wflow model settings file (.toml). You don't have to do it through HydroMT-Wflow.

In [ ]:
model.setup_config({'time.starttime': '2010-02-02T00:00:00', # Start time of our model
                    'time.endtime': '2010-02-10T00:00:00', # End time of our model
                    'time.timestepsecs': 86400, # Time step of our model in seconds
                    'output.netcdf_grid.path': 'gridded_output.nc', # Name of the NetCDF output file
                    'output.netcdf_grid.compressionlevel': 1, # Compression level for NetCDF output
                    'output.netcdf_grid.variables.land_surface__evapotranspiration_volume_flux': 'actual_evap' # Write actual evapotranspiration to a netCDF
                   })

### Basemaps
Let's begin by adding a basemap. Setting up the basemaps is always the first step, as it defines the geographic region of our model. Not only that; it also extracts some essential data from our DataCatalog and adds it to the model, such as land elevation, local drainage direction, and the slope. 

Note that we use the subbasin definition we checked earlier. It comes in handy now!

In [ ]:
model.setup_basemaps(region={'subbasin': [x, y], 'uparea': 70},
                     hydrography_fn='merit_hydro',
                     res=0.0083333) # resolution (unit is the same as the source CRS, so degrees in this case --> ~1 km)

Our model now has a basemap; let's have a look at the data it now contains!

In [ ]:
display(model.staticmaps.data)

Note that HydroMT-Wflow also stores the geometry of our delineated subbasin at model resolution, as well as at the original (higher) resolution. Let's plot our model region!

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Plot the elevation data from the model, masking out nodata values
model.staticmaps.data['land_elevation'].raster.mask_nodata().plot(
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=2500,
    cbar_kwargs={'label': 'Elevation [m]', 'shrink': 0.7})

model.geoms.data['basins'].plot(
    ax=ax, facecolor='none', edgecolor='black', label='Subbasin at model resolution')
model.geoms.data['meta_basins_highres'].plot(
    ax=ax, facecolor='none', edgecolor='red', label='High-resolution subbasin')
ax.set_xlabel('Longitude [degrees]')
ax.set_ylabel('Latitude [degrees]')
ax.set_title('Basemap of Wflow model')
ax.legend(loc='lower right')
ax.set_aspect('equal');

Note that we also could have used `region={'geom': gdf_basin}` instead of `region={'subbasin': [x, y], 'uparea': 70}`, reusing the basin delineation from earlier. However, using kind `geom` for `region` is not recommended, as it can lead to mistakes in the basin delineation.

### Rivers
The next step for building our model is to setup the rivers! We tell HydroMT-Wflow which datasets it should use, and it will look for those datasets in our DataCatalog, extract the relevant river data, and add it to our model.

To add the river data to our model, we need to define which cells contain a river. In HydroMT-Wflow, this is based on an upstream area threshold. Any cell that meets that threshold, is defined as a river cell and will have river data added.

It is crucial that we therefore use the same hydrography dataset for setting up the river as we had used to define our basemaps. Variables like the upstream area and local drainage direction were used to delineate our basemaps, and should therefore also be used to define which cells contain a river.

We also supply an extra dataset with river width and bankfull discharge data, as our hydrography dataset does not contain this data. This second dataset only contains data on rivers that are at least 30 meters wide. Therefore, we define a minimum river width, that will be used for cells where our second dataset has no data.

We also want to add the river depth, but we have no dataset with the river depth! As a work-around, we estimate the river depth based on the bankfull discharge (which we do have data on) and the power law, by specifying `rivdph_method='powlaw'`. But if you have data on the river depth, you could of course also just use that instead.

Lastly, we define the method to use for river routing, `river_routing='kinematic_wave'` in this case. 

In [ ]:
model.setup_rivers(
    hydrography_fn='merit_hydro', # Point to the entry in the data catalog containing the hydrography data
    river_geom_fn='hydro_rivers_lin', # river source data with river width and bankfull discharge
    river_upa=10, # The upstream area threshold in km2 to define a river
    rivdph_method='powlaw', # The method to determine the river depth. Here, the power law is used
    min_rivdph=1, # The minimum river depth in meters
    min_rivwth=10, # The minimum river width, in meters.
    river_routing='kinematic_wave') # The method to use for river routing

Let's look at the data we added!

In [ ]:
display(model.staticmaps.data)

Our model geometries now also contain the rivers. Let's plot these, using the stream order to set the width of the rivers:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Plot the elevation data from the model, masking out nodata values
model.staticmaps.data['land_elevation'].raster.mask_nodata().plot(
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=2500,
    cbar_kwargs={'label': 'Elevation [m]', 'shrink': 0.7})

model.geoms.data['basins'].plot(
    ax=ax, facecolor='none', edgecolor='black')
model.geoms.data['rivers'].plot(
    ax=ax, facecolor=None, edgecolor='blue', linewidth=model.geoms.data['rivers'].strord / 3)
ax.set_xlabel('Longitude [degrees]')
ax.set_ylabel('Latitude [degrees]')
ax.set_title('Basemap of Wflow model')
ax.set_aspect('equal');

Note that river data has of course only been added to cells with a river. For example, the river width:

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Plot the elevation data from the model, masking out nodata values
model.staticmaps.data['river_width'].raster.mask_nodata().plot(
    ax=ax,
    cmap='viridis_r',
    cbar_kwargs={'label': 'River width [m]', 'shrink': 0.7})

model.geoms.data['basins'].plot(
    ax=ax, facecolor='none', edgecolor='black', label='Subbasin at model resolution')
ax.set_xlabel('Longitude [degrees]')
ax.set_ylabel('Latitude [degrees]')
ax.set_title('River width')
ax.set_aspect('equal');

Let's also set the Manning roughness of the rivers. In HydroMT-Wflow, we use a mapping between Manning roughness and streamorder to determine the Manning roughness of rivers. HydroMT-Wflow contains a default mapping, which we will use here:

In [ ]:
model.setup_river_roughness()

### Reservoirs, lakes, and glaciers
Next, let's add controlled reservoirs, uncontrolled reservoirs (lakes), and glaciers. Again, our artifact data contains datasets with the necessary data for this. You could of course also supply your own datasets with the required data.

In [ ]:
model.setup_reservoirs_simple_control(
      reservoirs_fn= "hydro_reservoirs", # source for reservoirs shape and attributes
      timeseries_fn= "gww", # additional source for reservoir are timeseries to compute reservoirs, Either 'gww' using gwwapi or 'jrc' using hydroengine.
      min_area= 0.1, # minimum lake area to consider [km2]
)

In [ ]:
model.setup_reservoirs_no_control(
      reservoirs_fn= "hydro_lakes", # source for uncontrolled reservoirs (e.g. lakes) based on hydroLAKES: {hydro_lakes}
      min_area= 1.0, # minimum lake area to consider [km2]
)

In [ ]:
model.setup_glaciers(
    glaciers_fn= "rgi", # source for glaciers shape and attributes based on Randolph Glacier Inventory: {rgi}
    min_area= 0.1 # minimum glacier area to consider [km2]
)

Let's have a look at the data we just added to our model. You'll see that many variables related to glaciers and reservoirs have been added!

In [ ]:
model.staticmaps.data[
    [var for var in model.staticmaps.data if "reservoir" in var or "glacier" in var]]

Also note that our model has now extra geometries:

In [ ]:
for geom in model.geoms.data.keys():
    print(geom)

Note that we have stored a high resolution polygon of all reservoirs with simple control, and all reservoirs with no control. We also have a polygon containing all reservoirs (both with and without control) at model resolution. Also, a high-resolution polygon of the glaciers has been added. Let's plot these!

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Plot the elevation data from the model, masking out nodata values
model.staticmaps.data['land_elevation'].raster.mask_nodata().plot(
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=2500,
    cbar_kwargs={'label': 'Elevation [m]', 'shrink': 0.7})

model.geoms.data['basins'].plot(
    ax=ax, facecolor='none', edgecolor='black')
model.geoms.data['rivers'].plot(
    ax=ax, facecolor=None, edgecolor='blue', label='Rivers', linewidth=model.geoms.data['rivers'].strord / 3)
model.geoms.data['meta_reservoirs_no_control'].plot(
    ax=ax, facecolor='lightblue', edgecolor='black', zorder=4, label='Uncontrolled reservoirs')
model.geoms.data['meta_reservoirs_simple_control'].plot(
    ax=ax, facecolor='darkblue', edgecolor='darkblue', zorder=4, label='Controlled reservoirs')
model.geoms.data['glaciers'].plot(
    ax=ax, facecolor='grey', edgecolor='grey', zorder=4, label='Glaciers')
ax.set_xlabel('Longitude [degrees]')
ax.set_ylabel('Latitude [degrees]')
ax.set_title('Basemap of Wflow model')
ax.legend(loc='lower right')
ax.set_aspect('equal');

### Landuse
Next, let's setup the landuse-landcover (LULC) for our Wflow model. This is an important step, as many Wflow input variables are derived based on the landuse-landcover data. There are many different landuse maps available, and you can provide your own landuse map, with your own mapping between landuse classes and Wflow parameters. For several well-known landuse datasets, HydroMT-Wflow has a default mapping available.

In [ ]:
model.setup_lulcmaps(lulc_fn="corine")

This also adds quite a few layers to our Wflow model. Check-out how many variables our Wflow model already has:

In [ ]:
print(f'The Wflow model has {len(model.staticmaps.data.keys())} static layers.')

### Leaf area index
Next, we will add the leaf area index (LAI). We don't use a static LAI, but rather make use of a LAI climatology. That is to say, every month [1, 2, 3, ..., 12] gets a separate LAI. 

In [ ]:
model.setup_laimaps(lai_fn="modis_lai")

Because we have a separate value per month, the LAI that has been added to our Wflow model has 3 dimensions: x, y, and time, whereas other variables so far just had two dimensions (x and y). Nevertheless, we group the LAI under the "static" maps of our Wflow model.

In [ ]:
model.staticmaps.data['vegetation_leaf_area_index']

Let's have a quick look:

In [ ]:
lai = model.staticmaps.data["vegetation_leaf_area_index"]

fig, axes = plt.subplots(4, 3, figsize=(10, 10))
axes = axes.flatten()

for i in range(12):
    im = lai.isel(time=i).plot(
        ax=axes[i],
        add_colorbar=False,
        cmap="YlGn",
        vmin=0, vmax=5,
    )

    axes[i].set_title(f"Month {i+1}")
    axes[i].set_xlabel("")
    axes[i].set_ylabel("")
    axes[i].set_xticks([])
    axes[i].set_yticks([])
    axes[i].set_aspect("equal")

cbar = fig.colorbar(
    im,
    ax=axes,
    shrink=0.7,
    label="Leaf Area Index [m2/m2]",
)

plt.show()

### Soil maps
Next, let's setup the soilmaps. For this, we often use the SoilGrids dataset. Quite a few Wflow input parameters are derived based on this step! HydroMT-Wflow will use various pedo-transfer functions (PTF) to estimate these parameters, based on the soil composition. For the hydraulic conductivity, you can even choose which PTF you want to use!

In [ ]:
model.setup_soilmaps(soil_fn="soilgrids",
                     ptf_ksatver='brakensiek')

Our model has now even more layers in the staticmaps! However, note how our forcings are still empty, as we still haven't added them.

In [ ]:
print(f'The Wflow model has {len(model.staticmaps.data.keys())} '
      f'static layers and {len(model.forcing.data.keys())} forcing layers.')

### Meteorological forcings
Let's add some meteorological forcings to our model. Wflow requires three meteorological forcings: precipitation, temperature, and potential evapotranspiration.

Let's start with the precipitation. We will use a dataset from our data catalog, but you could very easily use your own dataset here. 

In [ ]:
model.setup_precip_forcing(precip_fn="chirps_global")

Let's take a look at the average precipitation!

In [ ]:
start = model.forcing.data["time"][0].dt.strftime("%Y-%m-%d").item()
end = model.forcing.data["time"][-1].dt.strftime("%Y-%m-%d").item()

model.forcing.data["precip"].mean(dim="time").plot(
    cbar_kwargs={"label": "Mean precipitation [mm/day]"}
)
plt.title(f"Mean precipitation between {start} and {end}")

In [ ]:
plt.bar(model.forcing.data['precip'].time,
        model.forcing.data['precip'].mean(dim=['x', 'y']),
        color='blue')
plt.ylabel('Mean precipitation [mm/day]')
plt.xticks(rotation=45);
plt.title('Spatially mean precipitation CHIRPS')

The spatial resolution of our precipitation dataset is coarser than our model resolution. Nevertheless, the precipitation data will be stored at model resolution.

You can try out a few different precipitation datasets available in the artifact data, and see what their resolution is, and how they compare spatially and temporally for this region and time period. Choosing the right precipitation dataset for your Wflow model is important, so take your time for this step! 

When it comes to precipitation, we always have to be a bit careful about the units. Precipitation is given in mm per timestep, in our case in mm/day. When setting up a Wflow model with HydroMT-Wflow, it is important to always check things like these, and confirm that the model makes sense.

We can now add the potential evapotranspiration (PET) and the temperature. These are derived in the same workflow, as the PET depends directly on the air temperature.

For the temperature, we can correct the temperature using the elevation lapse rate. For the PET, HydroMT-Wflow assumes the PET needs to be derived from other meteorological data, as direct PET data is often not available. There are multiple methods available to compute the PET, and the meteo variables required depend on the method chosen. Note that which method you choose also has a large effect on the eventual PET, so think carefully about this step.

Here, we will use the ERA5 dataset to derive the temperature and PET. The De Bruin method will be used to estimate the PET. The De Bruin method requires pressure, short wave incoming radiation, and TOA incident solar radiation. Just like the temperature, we can use an elevation lapse rate to correct the pressure.

For the lapse rate correction, it can help to provide a separate DEM, that has the same resolution as our original meteo data. However, if this is not provided, HydroMT-Wflow will fallback on the model DEM.

In [ ]:
model.setup_temp_pet_forcing(
    temp_pet_fn='era5',
    pet_method='debruin',
    temp_correction=True,
    press_correction=True,
    lapse_rate=-0.0065,
    dem_forcing_fn='era5_orography'
)

Let's take a look at the added temperature and PET!

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(8, 10))

# Temperature
model.forcing.data["temp"].mean(dim="time").plot(
    ax=axes[0],
    cbar_kwargs={"label": "Mean temperature [°C]"},
    vmin=-10,
    vmax=10,
    cmap="RdYlBu_r",
)
axes[0].set_title(f"Mean temperature between {start} and {end}")

# PET
model.forcing.data["pet"].mean(dim="time").plot(
    ax=axes[1],
    cbar_kwargs={"label": "Mean PET [mm/day]"},
    cmap='cividis'
)
axes[1].set_title(f"Mean PET between {start} and {end}")

axes[0].set_aspect('equal')
axes[1].set_aspect('equal')

plt.tight_layout()
plt.show()

ERA5 actually has quite a coarse resolution of 0.25 degrees. Nevertheless, due to the lapse rate correction, you can't directly see this in the resulting temperature and PET data. However, if you look carefully you can still see the outlines of the original ERA5 grid cells!

### Add model outlet and gauges
Wflow can output many, many different variables. If we are interested in a certain variable, we could of course have Wflow write output for that variable for every single grid cell. However, that is quite data intensive, and we are often interested in specific locations or areas. For this, we can make use of gauges in Wflow.

These are not necessarily actual gauges, but just points of interest in our model. We can have Wflow write model output for a variable at these gauges. This is for example handy when we are interested in river discharge at a specific location in our basin.

We can also derive the corresponding subcatchments from our model gauges. For these subcatchments, we can subsequently have Wflow write spatially aggregated data. For example, we could have Wflow write the mean/min/max groundwater recharge for each subcatchment.

With HydroMT-Wflow, we can add these handy gauges to our Wflow model. One "gauge" that we always want to include is the outlet of our model. Let's start with that.

In [ ]:
model.setup_outlets()

However, we might also be interested in some locations within our model domain. As an example, let's say we are interested in the locations defined in `test_stations.csv`:

In [ ]:
df_gauges = pd.read_csv("test_stations.csv").drop(columns='uparea')
df_gauges

Let's add these locations to our Wflow model as well. We can enforce that every gauge should be snapped to a river cell. Let's also derive the corresponding subcatchment for each gauge.

In [ ]:
model.setup_gauges(gauges_fn="test_stations.csv",
                   snap_to_river=True,
                   derive_subcatch=True)

Let's have a look at the resulting model!

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Plot the elevation data from the model, masking out nodata values
model.staticmaps.data['land_elevation'].raster.mask_nodata().plot(
    ax=ax,
    cmap=cmap,
    vmin=0,
    vmax=2500,
    cbar_kwargs={'label': 'Elevation [m]', 'shrink': 0.7})

model.geoms.data['basins'].plot(
    ax=ax, facecolor='none', edgecolor='black')
model.geoms.data['subcatchment_test-stations'].plot(
    ax=ax, facecolor='none', edgecolor='black')
model.geoms.data['rivers'].plot(
    ax=ax, facecolor=None, edgecolor='blue', label='Rivers', linewidth=model.geoms.data['rivers'].strord / 3)
model.geoms.data['meta_reservoirs_no_control'].plot(
    ax=ax, facecolor='lightblue', edgecolor='black', zorder=4, label='Uncontrolled reservoirs')
model.geoms.data['meta_reservoirs_simple_control'].plot(
    ax=ax, facecolor='darkblue', edgecolor='darkblue', zorder=4, label='Controlled reservoirs')
model.geoms.data['glaciers'].plot(
    ax=ax, facecolor='grey', edgecolor='grey', zorder=4, label='Glaciers')
model.geoms.data['gauges_test-stations'].plot(
    ax=ax, facecolor='orange', marker="^", zorder=4, label='Gauges')
model.geoms.data['outlets'].plot(
    ax=ax, facecolor='red', marker="o", zorder=4, label='Outlet')
ax.set_xlabel('Longitude [degrees]')
ax.set_ylabel('Latitude [degrees]')
ax.set_title('Basemap of Wflow model')
ax.legend(loc='lower right')
ax.set_aspect('equal');

Notice also the delineated subcatchments. For each gauge (including the outlet) we can have Wflow output point data, for example the river discharge at that location to a CSV or to a scalar netCDF. Furthermore, for each subcatchment (as well as the entire model domain), we can have Wflow output spatially aggregated data, for example the average actual evapotranspiration per subcatchment.

### Constant parameters
We are almost done setting up our Wflow model. There are still some Wflow input parameters we haven't added yet. For some parameters, we might not have any available data. Or, if a parameter can be estimated based on other data, we might not have the required auxiliary data available.

For these parameters we can set a constant value for the entire model domain. Nevertheless, each constant parameter is still added as 2D layer, with a value per grid cell.

In [ ]:
model.setup_constant_pars(
      subsurface_water__horizontal_to_vertical_saturated_hydraulic_conductivity_ratio=100,
      snowpack__degree_day_coefficient=3.75653,
      soil_surface_water__infiltration_reduction_parameter=0.038,
      vegetation_canopy_water__mean_evaporation_to_mean_precipitation_ratio=0.11,
      compacted_soil_surface_water__infiltration_capacity=10,
      soil_water_saturated_zone_bottom__max_leakage_volume_flux=0,
      soil_wet_root__sigmoid_function_shape_parameter=-500,
      atmosphere_air__snowfall_temperature_threshold=0,
      atmosphere_air__snowfall_temperature_interval=2,
      snowpack__melting_temperature_threshold=0,
      snowpack__liquid_water_holding_capacity=0.1,
      glacier_ice__degree_day_coefficient=3,
      glacier_firn_accumulation__snowpack_dry_snow_leq_depth_fraction=0.001,
      glacier_ice__melting_temperature_threshold=0)

### Writing our model to disk
We are finished setting up our model. At the start, we had set the model root to `./wflow_test_python_full`, but you might have noticed that this directory is still empty (save for perhaps some data HydroMT-Wflow downloaded for the reservoirs). That is because our entire model is still in memory. As a final step, let's write our model to disk.

In [ ]:
model.write()

For such a small model as this one, it will be pretty quick, but for larger model it might take a bit longer. Especially for models with a long temporal domain, writing the forcing data can take a while. 

You should now see several files in your model root, such as the `staticmaps.nc` (containing the static data), the `inmaps.nc` (containing forcing data), `wflow_sbm.toml` (setting of the model), `hydromt_data.yml` (a data catalog containing the data we used to build the model), and a directory with the model geometries (`/staticgeoms`). You can now run your model!